# Paired QM9 checkpoint validation

Run cells in order. This notebook uses recovered checkpoints and does not retrain.
It checks 256 preselected validation molecules, then compares EGNN, fusion with
clean topology, and fusion with topology from the same perturbed coordinates.
Sigma is 0 and 0.10 angstrom; targets stay the original gap values.

A standard CPU runtime is sufficient for the diagnostic. If a GPU runtime is
already selected, the notebook records it and uses it. Outputs are written to a
new dated Drive directory; historical artifacts remain unchanged. Inspect the
summary and recovery checks before deciding on expanded evaluation or training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, sys, subprocess, json, shutil, platform, time
from pathlib import Path
from datetime import datetime, timezone
REVISION = 'd939dfffbd6c9a21255ddd042c9b690d68840f99'
ORIGINAL = Path('/content/drive/MyDrive/qm9-egnn-tda')
assert (ORIGINAL / 'data/qm9/processed/data_v3.pt').is_file(), 'Locate the original Drive folder first'
STARTED = time.perf_counter()
RUN = ORIGINAL / 'validation' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN.mkdir(parents=True, exist_ok=False)
REPO = Path('/content/qm9-validation-' + REVISION[:8])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/serafim-tkachenko/qm9-egnn-tda.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == REVISION
GPU = shutil.which('nvidia-smi') is not None
hardware = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout if GPU else 'No GPU allocated'
(RUN / 'hardware-start.txt').write_text(hardware)
print(hardware)
print('Output directory:', RUN)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
UV = shutil.which('uv')
ENV = Path('/content/qm9-validation-env')
if not ENV.exists():
    subprocess.run([UV, 'venv', '--python', '3.11', str(ENV)], check=True)
PY = str(ENV / 'bin/python')
subprocess.run([UV, 'pip', 'install', '--python', PY, 'torch==2.10.0', '--index-url',
                'https://download.pytorch.org/whl/' + ('cu128' if GPU else 'cpu')], check=True)
subprocess.run([UV, 'pip', 'install', '--python', PY, '-r', str(REPO / 'requirements-validation.txt')], check=True)
(RUN / 'environment.txt').write_text(subprocess.check_output([PY, '-m', 'pip', 'freeze'], text=True)) if subprocess.run([PY, '-m', 'pip', '--version'], capture_output=True).returncode == 0 else (RUN / 'environment.txt').write_text(subprocess.check_output([UV, 'pip', 'freeze', '--python', PY], text=True))
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print('Environment ready:', PY, 'setup seconds:', time.perf_counter() - STARTED)


In [ ]:
def run_logged(arguments, filename):
    with (RUN / filename).open('w') as log:
        process = subprocess.Popen([PY, *arguments], cwd=REPO, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        if process.wait() != 0:
            raise RuntimeError('Check ' + str(RUN / filename) + '; no expansion after a failed gate')
try:
    run_logged(['-m', 'pytest', '-q'], 'tests.log')
    run_logged(['-m', 'scripts.prepare_paired', '--original-root', str(ORIGINAL),
                '--out', str(RUN / 'recovery'), '--device', 'cuda' if GPU else 'cpu'], 'recovery.log')
    run_logged(['-m', 'src.eval_paired', '--data', str(ORIGINAL / 'data/qm9/processed/data_v3.pt'),
                '--provenance', str(RUN / 'recovery/provenance.json'),
                '--split-manifest', str(RUN / 'recovery/split42.json'),
                '--cache', str(RUN / 'cache'), '--out', str(RUN / 'pilot'),
                '--device', 'cuda' if GPU else 'cpu', '--compute-environment', 'colab'], 'pilot.log')
    from IPython.display import display, Image
    display(Image(filename=str(RUN / 'pilot/paired_mae.png')))
    print((RUN / 'pilot/summary.json').read_text())
finally:
    (RUN / 'runtime.json').write_text(json.dumps({
        'source_revision': REVISION, 'wall_seconds_including_setup': time.perf_counter() - STARTED,
        'gpu_allocated': GPU, 'colab_credits_consumed': None,
        'credit_accounting': 'Read start/end balance or usage in Colab; no value inferred',
        'training_performed': False, 'output_directory': str(RUN)
    }, indent=2))
    print('Persistent output directory:', RUN)
    print('Inspect saved outputs, then disconnect and delete the runtime when finished.')


## Interpretation and stopping

A negative paired fusion-minus-EGNN difference favors fusion. The intervals
resample molecules and condition on these checkpoints. This is a development
pilot, not independent confirmation or training-seed uncertainty. If recovery
checks fail, inspect their saved records before continuing. A fixed-grid redesign
requires new descriptors and retraining. Do not substitute it into old weights.
